# ArcVox — GPU Engine Verification (Colab / Kaggle)

**Purpose:** prove the *pretrained, open-source* engines ArcVox ships with actually
produce commercial-grade output — on a free GPU, with **no model training**.

> You are **not** training anything here. These weights were already trained by
> Resemble AI (Chatterbox), OpenAI/SYSTRAN (Whisper) and the avatar labs. We just
> download them and run inference. That is exactly what the ArcVox backend does.

This notebook generates three real artifacts so you can judge quality yourself:
1. **TTS + zero-shot voice clone** (Chatterbox) → `.wav` files you can listen to
2. **Transcription** (faster-whisper `large-v3`) → text from the audio above
3. **Talking-head avatar** (SadTalker) → an `.mp4` of a photo lip-syncing the audio

> ⚠️ **Privacy note:** running here uploads your samples to Google/Kaggle infra.
> That's fine for *your own testing*, but it is **not** the private production
> deployment — for real users, run these same engines on hardware you control.


## 0 · Confirm you have a GPU
Runtime → Change runtime type → **GPU** (Colab), or enable the GPU accelerator (Kaggle).

In [ ]:
!nvidia-smi

In [ ]:
# Small helper so the same notebook works on Colab AND Kaggle for file uploads.
import os

def upload_one(prompt="Upload a file"):
    """Return a local path to an uploaded file on Colab, or guide Kaggle users."""
    try:
        from google.colab import files  # Colab
        print(prompt)
        up = files.upload()
        return list(up.keys())[0]
    except Exception:
        # Kaggle: add your file via the right-hand 'Add Data' panel, then point here.
        print("Kaggle detected (or no Colab uploader).")
        print("Add your file via the 'Add Data'/'Upload' panel, then set the path manually:")
        print("  e.g.  ref = '/kaggle/input/my-voice/sample.wav'")
        return None

IS_KAGGLE = os.path.exists("/kaggle")
print("Environment:", "Kaggle" if IS_KAGGLE else "Colab / other")

## 1 · Voice — Chatterbox TTS + zero-shot cloning

Chatterbox (MIT, Resemble AI) is the HD voice engine. First a base voice, then a
**zero-shot clone** from a short reference clip — no training, just a sample at
inference time. This is the exact capability `backend/engines/tts.py` uses.

In [ ]:
!pip install -q chatterbox-tts

In [ ]:
import torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS

cb = ChatterboxTTS.from_pretrained(device="cuda")

text = "This is ArcVox, running entirely on a GPU you control. No cloud API was called."
wav = cb.generate(text)
ta.save("arcvox_tts.wav", wav, cb.sr)
print("Saved arcvox_tts.wav")

from IPython.display import Audio
Audio("arcvox_tts.wav")

In [ ]:
# Zero-shot voice clone: upload a clean 10-30s sample of ONE speaker.
ref = upload_one("Upload a 10-30s voice sample (wav/mp3) to clone:")
# Kaggle users: set ref manually, e.g. ref = '/kaggle/input/.../sample.wav'

if ref:
    cloned = cb.generate(
        "Now I am speaking in the cloned voice. Same studio, your hardware, your data.",
        audio_prompt_path=ref,
    )
    ta.save("arcvox_clone.wav", cloned, cb.sr)
    print("Saved arcvox_clone.wav")
    from IPython.display import Audio
    display(Audio("arcvox_clone.wav"))
else:
    print("No reference set — skipping clone. Set `ref` to a file path and re-run.")

In [ ]:
# Free Chatterbox from VRAM before loading Whisper (16GB can't hold both).
import gc
del cb
gc.collect(); torch.cuda.empty_cache()
print("Chatterbox unloaded; VRAM freed.")

## 2 · Transcription — faster-whisper `large-v3`

The strongest module: `large-v3` matches or beats commercial transcription APIs.
We transcribe the cloned audio we just made (or fall back to the base TTS clip),
closing the loop end-to-end. Same engine as `backend/engines/stt.py`.

In [ ]:
!pip install -q faster-whisper

In [ ]:
from faster_whisper import WhisperModel

asr = WhisperModel("large-v3", device="cuda", compute_type="float16")

audio_in = "arcvox_clone.wav" if os.path.exists("arcvox_clone.wav") else "arcvox_tts.wav"
segments, info = asr.transcribe(audio_in, vad_filter=True, word_timestamps=True)

print(f"Detected language: {info.language} (p={info.language_probability:.2f})")
print("-" * 60)
for s in segments:
    print(f"[{s.start:6.2f} - {s.end:6.2f}]  {s.text.strip()}")

In [ ]:
# Free Whisper before the avatar engine.
import gc
del asr
gc.collect(); torch.cuda.empty_cache()
print("Whisper unloaded; VRAM freed.")

## 3 · Talking-head avatar — SadTalker  *(heavier, optional)*

SadTalker lip-syncs **one portrait photo** to the audio. ~8GB VRAM, fits a T4.
This is the most fragile install (specific torch/ffmpeg deps + weight downloads),
so run it last. If a cell errors, it's usually a dependency pin — the voice and
transcription verification above already stand on their own.

> Alternatives wired in the backend: MuseTalk, EchoMimic (~16GB, tight on a T4),
> LivePortrait. SadTalker is the most reliable to verify first.

In [ ]:
%cd /content 2>/dev/null || cd /kaggle/working
!git clone https://github.com/OpenTalker/SadTalker
%cd SadTalker
!pip install -q -r requirements.txt

In [ ]:
# Download the SadTalker checkpoints (~2GB).
!bash scripts/download_models.sh

In [ ]:
# Upload a front-facing portrait photo (jpg/png).
photo = upload_one("Upload a clear, front-facing portrait photo:")
# Kaggle: set manually e.g. photo = '/kaggle/input/.../face.png'

audio_path = "/content/arcvox_clone.wav"
if not os.path.exists(audio_path):
    audio_path = "/content/arcvox_tts.wav"
# On Kaggle the working dir differs; adjust if needed:
if not os.path.exists(audio_path):
    audio_path = "/kaggle/working/arcvox_clone.wav"

print("Using audio:", audio_path)
print("Using photo:", photo)

In [ ]:
# Run the lip-sync. Produces an .mp4 under ./results/
!python inference.py \
    --driven_audio "$audio_path" \
    --source_image "$photo" \
    --result_dir ./results \
    --still --preprocess full --enhancer gfpgan

In [ ]:
# Show the newest generated video.
import glob
from IPython.display import Video
vids = sorted(glob.glob("./results/**/*.mp4", recursive=True), key=os.path.getmtime)
print("Generated:", vids[-1] if vids else "none found")
Video(vids[-1], embed=True) if vids else print("No video produced — check the log above.")

## Appendix · Option B — drive the *live* ArcVox app from this GPU

Instead of testing engines individually, you can boot the whole ArcVox backend
here and point your local React frontend at it through a free tunnel. Heavier and
more fragile (the session is ephemeral), but lets you click through the real UI.

```bash
# 1. Bring your code in (push the branch first, or git clone your repo)
!git clone -b claude/arc-vox-analysis-fpn183 <YOUR_REPO_URL> app
%cd app/backend
!pip install -q -r requirements.txt

# 2. Configure for GPU engines + allow the tunnel origin
import os
os.environ["TTS_ENGINE"]   = "chatterbox"
os.environ["WHISPER_MODEL"] = "large-v3"
os.environ["AVATAR_ENGINE"] = "sadtalker"   # set SADTALKER_DIR too
os.environ["JWT_SECRET"]    = "change-me"
os.environ["CORS_ORIGINS"]  = "*"            # tighten for real use

# 3. Run the API in the background
import subprocess, time
subprocess.Popen(["python", "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8001"])
time.sleep(8)

# 4. Expose it with cloudflared (no signup) and copy the https URL it prints
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8001
```

Then run the frontend locally with `REACT_APP_BACKEND_URL=<the cloudflared https url>`.
Remember: this is for **testing only** — your customers' data must not run through
a free Colab/Kaggle session in production.


## What this proves

- ✅ **No training required** — pretrained MIT-licensed weights, inference only.
- ✅ **Voice quality is real** — listen to `arcvox_tts.wav` vs `arcvox_clone.wav`.
- ✅ **Transcription is commercial-grade** — `large-v3` output above.
- ✅ **Avatars work** — the generated `.mp4` is the open-source ceiling (honest:
  good, but still a step behind HeyGen).

**Next:** if zero-shot clone fidelity isn't enough for a premium tier, *then* we
add optional per-voice fine-tuning (hours on this same GPU) — never base-model
training.